In [3]:
# STEP4_4_fig_eigenvalues_bar.py
# 目的: 図4.4 (MDS固有値のスクリープロット) を「棒グラフ」で作成する。
#       OH (#FF6347) と FP (#00CED1) を比較表示。

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

# ==========================================
# 1. 設定
# ==========================================
RUN_ID = "20251130_153500"
ROOT_BASE = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# 入力ファイル (STEP 3.2.02 の出力)
INPUT_DIR_OH = os.path.join(ROOT_BASE, "sub", "02_mds_STEP3.2_signlessCorr", f"run_{RUN_ID}", "variables", "OH")
INPUT_DIR_FP = os.path.join(ROOT_BASE, "sub", "02_mds_STEP3.2_signlessCorr", f"run_{RUN_ID}", "variables", "FP")

FILE_OH = f"MDS_eigen_contributions_variables_OH_{RUN_ID}.csv"
FILE_FP = f"MDS_eigen_contributions_variables_FP_{RUN_ID}.csv"

# 出力先
OUTPUT_DIR = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}", "Chapter4_Figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# プロット設定
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Helvetica']
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 300

# ==========================================
# 2. データ読み込み
# ==========================================
path_oh = os.path.join(INPUT_DIR_OH, FILE_OH)
path_fp = os.path.join(INPUT_DIR_FP, FILE_FP)

if not os.path.exists(path_oh) or not os.path.exists(path_fp):
    print(f"[ERROR] Eigenvalue files not found.")
else:
    df_oh = pd.read_csv(path_oh)
    df_fp = pd.read_csv(path_fp)

    # ==========================================
    # 3. 図の作成 (棒グラフ)
    # ==========================================
    # 上位N次元までを表示
    TOP_N = 30  # 棒グラフだと本数が多いと見づらいので30程度推奨
    
    df_oh_plot = df_oh.head(TOP_N)
    df_fp_plot = df_fp.head(TOP_N)

    # データの準備
    x = np.arange(TOP_N) + 1  # 次元 (1, 2, ..., N)
    width = 0.4               # 棒の幅

    fig, ax = plt.subplots(figsize=(10, 6))

    # 棒グラフの描画 (OHを左、FPを右に配置)
    rects1 = ax.bar(x - width/2, df_oh_plot['percent']*100, width, 
                    label='OH (Material Name)', color='#FF6347', alpha=0.9, edgecolor='none')
    
    rects2 = ax.bar(x + width/2, df_fp_plot['percent']*100, width, 
                    label='FP (Fragment)', color='#00CED1', alpha=0.9, edgecolor='none')

    # 装飾
    ax.set_title("Figure 4.4: Eigenvalue Contribution Ratio (OH vs FP)", fontweight="bold")
    ax.set_xlabel("Dimension (Axis)")
    ax.set_ylabel("Contribution Ratio (%)")
    ax.set_xticks(x)
    
    # X軸ラベルの間引き (全部表示すると重なる場合)
    if TOP_N > 20:
        # 5刻みなどで表示
        labels = [str(i) if i==1 or i%5==0 else "" for i in x]
        ax.set_xticklabels(labels)
    else:
        ax.set_xticklabels(x)

    ax.legend()
    ax.grid(True, axis='y', linestyle=':', alpha=0.6)
    
    # 軸の範囲調整
    ax.set_xlim(0, TOP_N + 1)
    
    plt.tight_layout()
    
    # 保存
    save_path_png = os.path.join(OUTPUT_DIR, "Fig_4.4_MDS_Eigenvalues_Bar.png")
    save_path_pdf = os.path.join(OUTPUT_DIR, "Fig_4.4_MDS_Eigenvalues_Bar.pdf")
    plt.savefig(save_path_png)
    plt.savefig(save_path_pdf)
    plt.close()
    
    print(f"✅ Generated Figure 4.4 (Bar Chart): {save_path_png}")

✅ Generated Figure 4.4 (Bar Chart): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/Thesis_Figures/run_20251130_153500/Chapter4_Figures/Fig_4.4_MDS_Eigenvalues_Bar.png


In [4]:
# STEP4_4_generate_text.py
# 目的: 論文の4.2.3節で使用する「固有値の寄与率」に関する記述を、
#       計算結果CSVから自動生成して出力する。

import os
import pandas as pd

# ==========================================
# 1. 設定
# ==========================================
RUN_ID = "20251130_153500"
ROOT_BASE = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# 入力ファイル (Variablesの固有値)
INPUT_DIR_OH = os.path.join(ROOT_BASE, "sub", "02_mds_STEP3.2_signlessCorr", f"run_{RUN_ID}", "variables", "OH")
INPUT_DIR_FP = os.path.join(ROOT_BASE, "sub", "02_mds_STEP3.2_signlessCorr", f"run_{RUN_ID}", "variables", "FP")

FILE_OH = f"MDS_eigen_contributions_variables_OH_{RUN_ID}.csv"
FILE_FP = f"MDS_eigen_contributions_variables_FP_{RUN_ID}.csv"

# ==========================================
# 2. データ読み込み & 計算
# ==========================================
path_oh = os.path.join(INPUT_DIR_OH, FILE_OH)
path_fp = os.path.join(INPUT_DIR_FP, FILE_FP)

def get_eigen_stats(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    
    # 寄与率 (percent列は 0.215... のようになっている想定 -> 100倍する)
    # csvの中身: axis, eigenvalue, percent, cum_percent
    
    p1 = df.loc[0, 'percent'] * 100
    p2 = df.loc[1, 'percent'] * 100
    p3 = df.loc[2, 'percent'] * 100
    
    cum2 = p1 + p2
    cum3 = p1 + p2 + p3
    
    return p1, p2, p3, cum2, cum3

stats_oh = get_eigen_stats(path_oh)
stats_fp = get_eigen_stats(path_fp)

if stats_oh is not None and stats_fp is not None:
    oh_p1, oh_p2, oh_p3, oh_cum2, oh_cum3 = stats_oh
    fp_p1, fp_p2, fp_p3, fp_cum2, fp_cum3 = stats_fp

    # ==========================================
    # 3. 文章生成
    # ==========================================
    # 最後の段落の「27.17%」の部分は、文脈的に「OHの累積寄与率」を指していると考えられます。
    # (OHは寄与率が低いため可視化が難しく、クラスタリングが必要という論理)
    # そのため、最後の {} にも oh_cum2 を入れています。

    text = f"""
ワンホットデーターを用いた第1軸と第2軸では、全体の固有値の{oh_cum2:.2f}%(第1軸{oh_p1:.2f}%; 第2軸{oh_p2:.2f}%)、第3軸まで加えると{oh_cum3:.2f}％であった。
また、フィンガープリントデーターを用いた第1軸と第2軸では、全体の固有値の{fp_cum2:.2f}%(第1軸{fp_p1:.2f}%; 第2軸{fp_p2:.2f}%)、第3軸まで加えると{fp_cum3:.2f}％であった。

　　図4.5では、多次元尺度構成法の第1軸と第2軸で変数間の関係を表した。このようにして、多次元尺度構成法を用いることにより、散布図としておおまかに変数間の関係を理解することができる。一方で、第1軸と第2軸では、固有値の合計の{oh_cum2:.2f}%であるため、厳密な意味での変数間の関係を把握するのは難しい。そこで、次節ではクラスター分析により変数を適切なグループに分類し検討する。
"""
    
    print("-" * 30)
    print("▼ 以下のテキストをコピーして論文に使用してください ▼")
    print("-" * 30)
    print(text)
    print("-" * 30)

else:
    print("[ERROR] Could not load eigenvalue files.")

------------------------------
▼ 以下のテキストをコピーして論文に使用してください ▼
------------------------------

ワンホットデーターを用いた第1軸と第2軸では、全体の固有値の10.22%(第1軸6.15%; 第2軸4.07%)、第3軸まで加えると13.93％であった。
また、フィンガープリントデーターを用いた第1軸と第2軸では、全体の固有値の22.72%(第1軸12.06%; 第2軸10.67%)、第3軸まで加えると29.97％であった。

　　図4.5では、多次元尺度構成法の第1軸と第2軸で変数間の関係を表した。このようにして、多次元尺度構成法を用いることにより、散布図としておおまかに変数間の関係を理解することができる。一方で、第1軸と第2軸では、固有値の合計の10.22%であるため、厳密な意味での変数間の関係を把握するのは難しい。そこで、次節ではクラスター分析により変数を適切なグループに分類し検討する。

------------------------------
